In [9]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["NO_PROXY"] = "localhost,127.0.0.1,0.0.0.0"
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

# 提前把千问大模型和 Embedding 雷达加载到内存中待命
llm = OllamaLLM(model="qwen2.5:0.5b") 
embedding_radar = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

print("✅ 后勤装配完毕！雷达与大模型已在内存中全天候待命！")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 后勤装配完毕！雷达与大模型已在内存中全天候待命！


In [10]:
target_filename = "secret_doc.txt"  # 👈 把它换成您真实的文件名

# 2. 检查文件是否存在并读取
if os.path.exists(target_filename):
    print(f"📡 发现目标文件：{target_filename}，正在加载内容...")
    
    # 采用标准读取模式
    loader = TextLoader(target_filename, encoding="utf-8")
    documents = loader.load()
    
    # 3. 撕碎文本并建立向量数据库
    # 这里保持不变，依然使用之前加载好的内存组件
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
    chunks = text_splitter.split_documents(documents)

    # 存入数据库
    vector_db = Chroma.from_documents(chunks, embedding_radar)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4}) 

    print(f"✅ 情报入库完毕！已将 {len(chunks)} 个碎片锁入雷达范围。")
else:
    print(f"❌ 警报：在当前目录下未找到文件【{target_filename}】！请核对路径。")

📡 发现目标文件：secret_doc.txt，正在加载内容...
✅ 情报入库完毕！已将 78 个碎片锁入雷达范围。


In [11]:
# 1. 准备机密文本 (可以随时在这里修改文本内容)


# 2. 撕碎并建立向量数据库
loader = TextLoader("secret_doc.txt", encoding="utf-8")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

# 存入 Chroma 数据库
vector_db = Chroma.from_documents(chunks, embedding_radar)
# 注意：把搜索范围 k 调大到 4，确保不会漏掉情报！
retriever = vector_db.as_retriever(search_kwargs={"k": 4}) 

print("✅ 机密资料已向量化并锁入数据库！检索雷达已上线！")

✅ 机密资料已向量化并锁入数据库！检索雷达已上线！


In [12]:
# 👉 在这里输入您的问题：
user_question = "黄仁勋讲了什么，总结一下。同时你觉得2026年的AI硬件市场，除了英伟达，谁还能打？"

# 1. 雷达瞬间检索
retrieved_docs = retriever.invoke(user_question)

# 2. 组装提示词
prompt_template = """你是一位资深的科技战略分析师。
请结合提供的<背景参考资料>，用专业的口吻回答指挥官的问题。

【指令规范】：
1. 优先使用资料中的数据（如性能百分比、具体代号）。
2. 在资料基础上，你可以进行合理的逻辑总结和技术关联。
3. 如果资料中完全没有提及某个关键点，请明确指出。

<背景参考资料>
{context}
</机密情报>

指挥官的问题：{question}
深度战术分析："""
prompt = PromptTemplate.from_template(prompt_template)
context_text = "\n".join([doc.page_content for doc in retrieved_docs])
final_prompt = prompt.format(context=context_text, question=user_question)

# 3. 千问副官开火
print(f"收到指令：【{user_question}】\n")
response = llm.invoke(final_prompt)
print("千问 副官回答：")
print(response)

收到指令：【黄仁勋讲了什么，总结一下。同时你觉得2026年的AI硬件市场，除了英伟达，谁还能打？】

千问 副官回答：
【指令】：
1. 提取《星野峰》的代号。
2. 识别表格中的数据，并进行计算或整合。

首先，提取《星野峰》的代号：

- Starfield, The Elder Scrolls IV: Oblivion Remastered, Where Winds Meet, and more.
- Starfield, The Elder Scrolls IV: Oblivion Remastered, Where Winds Meet, and more.
- Starfield, The Elder Scrolls IV: Oblivion Remastered, Where Winds Meet, and more.
- Starfield, The Elder Scrolls IV: Oblivion Remastered, Where Winds Meet, and more.

结合这些代号，我们可以了解到《星野峰》是EA开发的，《星野峰》系列共有四个版本：

- 原版：Starfield, The Elder Scrolls IV: Oblivion
- 重制版1（E3版）：The Elder Scrolls V: Skyrim
- 重制版2（OBC版）：The Elder Scrolls V: Skyrim Reborn
- 重制版3（RPG版）：The Elder Scrolls V: Skyrim Rebirth
